In [16]:
import json
import csv
import math
import time
import warnings
from pathlib import Path
from langchain_chroma import Chroma

DOCS_DIR        = "../../data/parsed_json/"
CHROMA_BASE     = "../../data/vector_db_prefix_1000"
GOLDEN_SET_PATH = "../../eval/golden_dataset/golden_dataset_v2.csv"
COLLECTION_NAME = "rfp_docs"
EMBEDDING_MODEL = "bge-m3"
BATCH_SIZE      = 500
TOP_K           = 10
CHUNK_SIZE      = 1000

In [2]:
STRATEGIES = [
    {"name": "A_full",  "prefix": "full"},  # 사업명+발주기관+요약
    {"name": "A_short", "prefix": "short"},  # 사업명+발주기관
    {"name": "A_none",  "prefix": "none"},  # prefix 없음
    {"name": "A_summary", "prefix": "summary"},  # 요약 별도 청크
]

GS_TO_DOCID = {
    "GKL_그룹웨어":            "D093",
    "KUSF_체육":               "D011",
    "강릉어선안전":             "D024",
    "경기_사회서비스":          "D087",
    "고려대학교_차세대포털":    "D008",
    "광주과기원_RCMS":          "D073",
    "광주과학기술원_학사시스템": "D039",
    "구미_육상":               "D018",
    "국립중앙의료원_응급":      "D069",
    "국민연금공단_이러닝":      "D049",
    "국민연금_멀티턴1":         "D050",
    "국민연금_멀티턴2":         "D050",
    "국민연금_멀티턴3":         "D050",
    "국방_대용량":             "D010",
    "기초과학연구원_극저온":    "D051",
    "나노종합_팹":             "D099",
    "대검찰청_홈페이지":        "D053",
    "민속박물관_아카이브":      "D090",
    "벤처협회_시스템":          "D086",
    "보험개발원_실손":          "D083",
    "봉화군_재난":             "D005",
    "부산관광_ERP":            "D037",
    "서민금융_채팅":            "D056",
    "서영대_교육":             "D045",
    "서울_디지털성범죄":        "D068",
    "서울_지도플랫폼":          "D040",
    "서울교육청_ISP":           "D084",
    "세종_인사":               "D088",
    "우즈벡_관개":             "D072",
    "울산_버스":               "D034",
    "인천_도시계획":            "D004",
    "인천_일자리":             "D030",
    "인천공항_ERP":            "D079",
    "적십자_재해복구":          "D095",
    "철도_ISP":               "D070",
    "통합정보시스템_충돌":      "D016",
    "평택_버스":               "D060",
    "해양박물관_자료":          "D066",
    "고려대_vs_광주과기원":     ["D008", "D039"],
    "버스_다중비교":            ["D034", "D060"],
    "재난안전_종합":            ["D005", "D007"],
    "철도_vs_인천_ISP":        ["D070", "D030"],
    "TEST": None, "unknown": None, "ISP_다중비교": None,
    "교육관련_다중문서": None, "문화_다중비교": None,
    "의료_다중문서": None, "재공고_종합": None,
    "신규_vs_고도화": None, "예산_최소_최대": None,
    "멀티턴_심화1": None, "멀티턴_심화2": None,
    "모른다_테스트1": None, "모른다_테스트2": None,
    "모른다_테스트3": None, "모른다_테스트4": None,
    "모른다_테스트5": None, "모른다_테스트6": None,
    "존재하지않는사업": None, "입찰마감_확인": None,
}


In [3]:
def clean(val):
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return val

In [4]:
def build_payload(doc: dict, section: dict, block: dict) -> dict:
    meta = doc.get("metadata", {})
    return {
        "doc_id":        str(clean(doc.get("doc_id"))),
        "file_name":     str(clean(doc.get("file_name"))),
        "source_format": str(clean(doc.get("source_format"))),
        "사업명":         str(clean(meta.get("사업명"))),
        "발주기관":       str(clean(meta.get("발주기관"))),
        "사업유형":       str(clean(meta.get("사업유형"))),
        "사업금액":       float(clean(meta.get("사업금액")) or 0.0),
        "공고번호":       str(clean(meta.get("공고번호"))),
        "공고차수":       float(clean(meta.get("공고차수")) or 0.0),
        "공개일자":       str(clean(meta.get("공개일자"))),
        "입찰참여시작일":  str(clean(meta.get("입찰참여시작일"))),
        "입찰참여마감일":  str(clean(meta.get("입찰참여마감일"))),
        "재공고여부":     bool(meta.get("재공고여부", False)),
        "linked_doc_id": str(clean(meta.get("linked_doc_id"))),
        "사업요약":       str(clean(meta.get("사업요약"))),
        "header_path":   " > ".join(section.get("header_path", [])),
        "section_id":    str(clean(section.get("section_id"))),
        "block_id":      str(clean(block.get("block_id"))),
        "block_type":    str(clean(block.get("type"))),
        "table_type":    str(clean(block.get("table_type"))),
    }


In [5]:
def chunk_section(section: dict) -> list[dict]:
    header_prefix = " > ".join(section.get("header_path", []))
    results = []
    current_text = ""
    last_text_block = None

    for block in section.get("blocks", []):
        if block.get("is_decorative"):
            continue
        if block["type"] == "table":
            if current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = ""
                last_text_block = None
            results.append({
                "content": f"[섹션: {header_prefix}]\n\n{block['content']}",
                "block":   block,
            })
        else:
            if len(current_text) + len(block["content"]) > CHUNK_SIZE and current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = block["content"] + "\n\n"
            else:
                current_text += block["content"] + "\n\n"
            last_text_block = block

    if current_text.strip():
        results.append({
            "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
            "block":   last_text_block,
        })
    return results

In [6]:
def build_prefix(meta: dict, prefix_type: str) -> str:
    name     = str(clean(meta.get("사업명")))
    org      = str(clean(meta.get("발주기관")))
    summary  = str(clean(meta.get("사업요약")))

    if prefix_type == "full":
        return f"[사업명] {name}\n[발주기관] {org}\n[요약] {summary}\n\n"
    elif prefix_type in ("short", "summary"):
        return f"[사업명] {name}\n[발주기관] {org}\n\n"
    else:
        return ""

In [7]:
def process_doc(doc: dict, prefix_type: str) -> tuple[list[str], list[dict]]:
    texts, metas = [], []
    meta = doc.get("metadata", {})
    prefix = build_prefix(meta, prefix_type)

    # 요약 별도 청크 추가
    if prefix_type == "summary":
        name    = str(clean(meta.get("사업명")))
        org     = str(clean(meta.get("발주기관")))
        summary = str(clean(meta.get("사업요약")))
        if summary.strip():
            texts.append(f"[사업명] {name}\n[발주기관] {org}\n[요약] {summary}")
            metas.append(build_payload(doc, {}, {}))

    for section in doc.get("sections", []):
        chunks = chunk_section(section)
        for item in chunks:
            texts.append(prefix + item["content"])
            metas.append(build_payload(doc, section, item["block"] or {}))

    return texts, metas

In [8]:
def load_embedding_model(name: str):
    if name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "text-embedding-3-small":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model="text-embedding-3-small")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")

In [9]:
def save_vectorstore(strategy: dict, all_texts: list, all_metas: list, embedding_model):
    chroma_dir = f"{CHROMA_BASE}/{strategy['name']}"

    if Path(chroma_dir).exists():
        vs = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=chroma_dir,
        )
        existing_count = vs._collection.count()
        if existing_count >= len(all_texts):
            print(f"[SKIP] {strategy['name']} — 완료 ({existing_count}개 청크)")
            return vs
        print(f"[RESUME] {strategy['name']} — {existing_count}/{len(all_texts)}개, 이어서 진행")
        start_from = existing_count
    else:
        vs = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embedding_model,
            persist_directory=chroma_dir,
        )
        start_from = 0

    for start in range(start_from, len(all_texts), BATCH_SIZE):
        end = start + BATCH_SIZE
        vs.add_texts(
            texts=all_texts[start:end],
            metadatas=all_metas[start:end],
        )
        print(f"  저장 완료: {min(end, len(all_texts))}/{len(all_texts)}")

    print(f"  완료 ({vs._collection.count()}개 청크) → {chroma_dir}")
    return vs

In [10]:
def hit(retrieved_ids, target_ids, k):
    return any(tid in retrieved_ids[:k] for tid in target_ids)

In [11]:
# 골든셋 로드
golden_set = []
with open(GOLDEN_SET_PATH, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        golden_set.append(row)

golden_set = golden_set[:101]
print(f"골든셋 문항 수: {len(golden_set)}")

json_files = sorted(Path(DOCS_DIR).glob("*.json"))
print(f"JSON 파일 수: {len(json_files)}")

embedding_model = load_embedding_model(EMBEDDING_MODEL)
warnings.filterwarnings("ignore", message="Relevance scores must be between")

골든셋 문항 수: 101
JSON 파일 수: 97


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [17]:
# 전략별 실험 
eval_results = []

for strategy in STRATEGIES:
    print(f"\n{'='*50}")
    print(f"[전략] {strategy['name']} (prefix={strategy['prefix']})")

    all_texts, all_metas = [], []
    for json_file in json_files:
        with open(json_file, encoding="utf-8") as f:
            doc = json.load(f)
        texts, metas = process_doc(doc, prefix_type=strategy["prefix"])
        all_texts.extend(texts)
        all_metas.extend(metas)

    print(f"  총 청크: {len(all_texts)}개 | 평균 길이: {sum(len(t) for t in all_texts)//len(all_texts)}자")

    vs = save_vectorstore(strategy, all_texts, all_metas, embedding_model)

    recall3, recall5, recall10, mrr_scores, query_times = [], [], [], [], []
    skipped = 0

    for row in golden_set:
        gs_key = str(row["doc_id"])
        target = GS_TO_DOCID.get(gs_key)

        if target is None:
            skipped += 1
            continue

        target_ids = target if isinstance(target, list) else [target]

        t0 = time.time()
        hits = vs.similarity_search_with_relevance_scores(row["question"], k=TOP_K)
        query_times.append(time.time() - t0)

        retrieved_doc_ids = [doc.metadata.get("doc_id", "") for doc, _ in hits]

        recall3.append(1.0 if hit(retrieved_doc_ids, target_ids, 3) else 0.0)
        recall5.append(1.0 if hit(retrieved_doc_ids, target_ids, 5) else 0.0)
        recall10.append(1.0 if hit(retrieved_doc_ids, target_ids, 10) else 0.0)

        rank = next(
            (i + 1 for i, d in enumerate(retrieved_doc_ids) if d in target_ids),
            None,
        )
        mrr_scores.append(1.0 / rank if rank else 0.0)

    n = len(recall3)
    print(f"  평가: {n}개 | 제외: {skipped}개")

    eval_results.append({
        "전략":       strategy["name"],
        "prefix":     strategy["prefix"],
        "Recall@3":   round(sum(recall3)    / n, 4),
        "Recall@5":   round(sum(recall5)    / n, 4),
        "Recall@10":  round(sum(recall10)   / n, 4),
        "MRR":        round(sum(mrr_scores) / n, 4),
        "avg_ms":     round(sum(query_times) / n * 1000, 1),
    })



[전략] A_full (prefix=full)
  총 청크: 17382개 | 평균 길이: 728자
[SKIP] A_full — 완료 (17382개 청크)
  평가: 84개 | 제외: 17개

[전략] A_short (prefix=short)
  총 청크: 17382개 | 평균 길이: 461자
[SKIP] A_short — 완료 (17382개 청크)
  평가: 84개 | 제외: 17개

[전략] A_none (prefix=none)
  총 청크: 17382개 | 평균 길이: 406자
[SKIP] A_none — 완료 (17382개 청크)
  평가: 84개 | 제외: 17개

[전략] A_summary (prefix=summary)
  총 청크: 17479개 | 평균 길이: 460자
[SKIP] A_summary — 완료 (17479개 청크)
  평가: 84개 | 제외: 17개


In [14]:
# 결과 출력 
from IPython.display import Markdown, display

header = "| 전략 | prefix | Recall@3 | Recall@5 | Recall@10 | MRR | avg_ms |\n|------|--------|----------|----------|-----------|-----|--------|"
rows = "\n".join(
    f"| {r['전략']} | {r['prefix']} | {r['Recall@3']} | {r['Recall@5']} | {r['Recall@10']} | {r['MRR']} | {r['avg_ms']} |"
    for r in eval_results
)
display(Markdown(f"## prefix 전략 실험 결과\n\n{header}\n{rows}"))

## prefix 전략 실험 결과

| 전략 | prefix | Recall@3 | Recall@5 | Recall@10 | MRR | avg_ms |
|------|--------|----------|----------|-----------|-----|--------|
| A_full | full | 0.1905 | 0.2024 | 0.2143 | 0.1793 | 25.0 |
| A_short | short | 0.2262 | 0.2857 | 0.3095 | 0.2238 | 25.2 |
| A_none | none | 0.1429 | 0.1667 | 0.25 | 0.1412 | 25.5 |
| A_summary | summary | 0.2619 | 0.2738 | 0.3095 | 0.2421 | 25.2 |

A_summary > A_short > A_full > A_none

**1. 요약은 반복보다 분리가 효과적**
- A_full은 요약을 매 청크마다 붙여서 오히려 성능 저하
- A_summary는 요약을 별도 청크로 분리해 사업 개요 질문에서 직접 hit되고, 나머지 청크는 실제 내용에 집중할 수 있어 변별력 향상
- Recall@3 기준 A_full(0.1905) vs A_summary(0.2619) → 37% 향상

**2. 최소한의 식별자는 필요**
- A_none은 prefix가 전혀 없어 사업명/발주기관 정보 없이 내용만으로 검색
- 비슷한 내용의 청크들 사이에서 문서 구분이 어려워 가장 낮은 성능
- A_short 대비 Recall@3 기준 37% 낮음

**3. prefix는 짧을수록 좋음**
- A_short(사업명+발주기관)가 A_full(사업명+발주기관+요약)보다 전 지표 우세
- 긴 요약이 매 청크에 반복되면 임베딩 공간에서 모든 청크가 유사해져 검색 품질 저하

### 지표 비교

| 전략 | Recall@3 | Recall@5 | Recall@10 | MRR |
|------|----------|----------|-----------|-----|
| A_full | 0.1905 | 0.2024 | 0.2143 | 0.1793 |
| A_short | 0.2262 | 0.2857 | 0.3095 | 0.2238 |
| A_none | 0.1429 | 0.1667 | 0.2500 | 0.1412 |
| A_summary | 0.2619 | 0.2738 | 0.3095 | 0.2421 |

### 결론
**A_summary 전략 채택** — 요약을 별도 청크로 분리하고 나머지 청크엔 사업명+발주기관만 붙이는 방식이 retrieval 정확도와 순위 품질 모두에서 최고 성능. chunk=500 실험에서도 동일한 경향이 나오면 최종 전략으로 확정.